# LLM-Powered Applications & Distributed Computing

## Part 1: Distributed Data Processing with Spark


In [1]:
from urllib.request import urlretrieve
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

files = [
    ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", raw_dir/"yellow_taxi_data.parquet"),
    ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", raw_dir/"taxi_zone_lookup.csv"),
]

for url, filename in files:
    urlretrieve(url, filename)

print("Done!")

Done!


### Spark Environment Setup & Data Loading

In [1]:
# Creating a SparkSession 
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("NYC Yellow Taxi Trip Records (January 2024)") \
    .master("local[*]") \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

In [2]:
# Verify the session
print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')

Spark version: 4.1.1
App name: NYC Yellow Taxi Trip Records (January 2024)
Master: local[*]
Default parallelism: 12


In [3]:
sc = spark.sparkContext  # Access SparkContext
print(sc.getConf().getAll())  # Print all configurations

[('spark.rdd.compress', 'True'), ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'), ('spark.app.submitTime', '1774064285869'), ('spark.executor.memory', '1g'), ('spark.sql.artifact.isolation.enabled', 'false'), ('spark.executor.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-

In [4]:
# Loading the NYC Yellow Taxi Parquet data into a Spark DataFrame
file_path = "data\\raw\\yellow_taxi_data.parquet"
df = spark.read.format("parquet").load(file_path)
df.show(5)
df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [5]:
print(f"Total rows: {df.count()}")
print(f"Total partitions: {df.rdd.getNumPartitions()}")

Total rows: 2964624
Total partitions: 12


In [18]:
import time
import pandas as pd

# Time Spark read (lazy - just metadata)
start = time.time()
df_spark = spark.read.parquet('data/raw/yellow_taxi_data.parquet')
spark_read_time = time.time() - start

# Time Spark action (forces full read)
start = time.time()
spark_count = df_spark.count()

spark_action_time = time.time() - start

# Time Pandas read
start = time.time()
df_pandas = pd.read_parquet('data/raw/yellow_taxi_data.parquet')
pandas_read_time = time.time() - start
print(f'Spark schema read: {spark_read_time:.3f}s (lazy - no data loaded)')
print(f'Spark count action: {spark_action_time:.3f}s ({spark_count:,} rows)')
print(f'Pandas full read: {pandas_read_time:.3f}s ({len(df_pandas):,} rows)')
print(f'Pandas memory usage: {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f}MB')

# Clean up Pandas DataFrame to free memory
del df_pandas

Spark schema read: 0.168s (lazy - no data loaded)
Spark count action: 0.289s (2,964,624 rows)
Pandas full read: 0.696s (2,964,624 rows)
Pandas memory usage: 418.0MB


Interpretation: Spark usually showed a lower load time than Pandas since it uses lazy evaluation unlike Pandas. So, in other words when Spark's load function is called it builds the plan rather than read the data immediately (only if an action is performed), whereas Pandas reads the entire files into memory right away.

### Data Cleaning & Feature Engineering in Spark

In [23]:
from pyspark.sql import functions as F

trips = df.select(
    F.col('tpep_pickup_datetime').alias('pickup_time'),
    F.col('tpep_dropoff_datetime').alias('dropoff_time'),
    'passenger_count',
    'trip_distance',
    'fare_amount',
    'tip_amount',
    'total_amount',
    'payment_type',
    'PULocationID',
    'DOLocationID'
)

print(f'Original rows: {df.count():,}')

Original rows: 2,964,624


In [25]:
df_nulls = trips.filter(
    F.col("pickup_time").isNull() |
    F.col("dropoff_time").isNull() |
    F.col("PULocationID").isNull() |
    F.col("DOLocationID").isNull() |
    F.col("fare_amount").isNull() |
    F.col("trip_distance").isNull()
)
df_nulls.show()

+-----------+------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+
|pickup_time|dropoff_time|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|PULocationID|DOLocationID|
+-----------+------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+
+-----------+------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+



In [26]:
# Remove rows with nulls in critical columns using Spark DataFrame API
critical_cols = [
    'pickup_time', 
    'dropoff_time', 
    'PULocationID', 
    'DOLocationID', 
    'fare_amount', 
    'trip_distance'
 ]

trips_clean = trips.dropna(subset=critical_cols)
print(f"Rows after cleaning: {trips_clean.count()}")
print(f"Number of rows removed: {trips.count() - trips_clean.count()}")

Rows after cleaning: 2964624
Number of rows removed: 0


In [ ]:
# Filter out invalid trips using Spark DataFrame API
trips_valid = trips_clean.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") >= 0) &
    (F.col("fare_amount") <= 500) &
    (F.col("dropoff_time") >= F.col("pickup_time"))
 )
print(f"Rows after filtering invalid trips: {trips_valid.count()}")
print(f"Number of rows removed: {trips_clean.count() - trips_valid.count()}")

Rows after filtering invalid trips: 2870102
Number of rows removed: 94522


In [ ]:
# # create new column for trip duration in minutes
# from pyspark.sql.functions import unix_timestamp, round as spark_round, col

# trips_enriched = trips_valid.withColumn(
#     "trip_duration_minutes",
#     spark_round(
#         (unix_timestamp(col("dropoff_time")) - unix_timestamp(col("pickup_time"))) / 60, 2
#     )
# )
 
# trips_enriched.select("pickup_time", "dropoff_time", "trip_duration_minutes").show(5)

+-------------------+-------------------+---------------------+
|        pickup_time|       dropoff_time|trip_duration_minutes|
+-------------------+-------------------+---------------------+
|2024-01-01 00:57:55|2024-01-01 01:17:43|                 19.8|
|2024-01-01 00:03:00|2024-01-01 00:09:36|                  6.6|
|2024-01-01 00:17:06|2024-01-01 00:35:01|                17.92|
|2024-01-01 00:36:38|2024-01-01 00:44:56|                  8.3|
|2024-01-01 00:46:51|2024-01-01 00:52:57|                  6.1|
+-------------------+-------------------+---------------------+
only showing top 5 rows


In [ ]:
# # create new column for trip speed mph
# from pyspark.sql.functions import col, when, round as spark_round

# trips_enriched = trips_enriched.withColumn(
#     "trip_speed_mph",
#     when(col("trip_duration_minutes") > 0, 
#          spark_round((col("trip_distance") / (col("trip_duration_minutes") / 60)), 2)).otherwise(0)
# )

# trips_enriched.select("trip_distance", "trip_duration_minutes", "trip_speed_mph").show(5)

+-------------+---------------------+--------------+
|trip_distance|trip_duration_minutes|trip_speed_mph|
+-------------+---------------------+--------------+
|         1.72|                 19.8|          5.21|
|          1.8|                  6.6|         16.36|
|          4.7|                17.92|         15.74|
|          1.4|                  8.3|         10.12|
|          0.8|                  6.1|          7.87|
+-------------+---------------------+--------------+
only showing top 5 rows


In [ ]:
# # create a new column for pickup_hour (0-23) and pickup_day_of_week (1=Sunday, 7=Saturday)
# from pyspark.sql.functions import hour, dayofweek, col

# trips_enriched = trips_enriched.withColumn("pickup_hour", hour(col("pickup_time"))) \
#     .withColumn("pickup_day_of_week", dayofweek(col("pickup_time"))
#     )

# trips_enriched.select("pickup_time", "pickup_hour", "pickup_day_of_week").show(5)

+-------------------+-----------+------------------+
|        pickup_time|pickup_hour|pickup_day_of_week|
+-------------------+-----------+------------------+
|2024-01-01 00:57:55|          0|                 2|
|2024-01-01 00:03:00|          0|                 2|
|2024-01-01 00:17:06|          0|                 2|
|2024-01-01 00:36:38|          0|                 2|
|2024-01-01 00:46:51|          0|                 2|
+-------------------+-----------+------------------+
only showing top 5 rows


In [ ]:
# # create a new column for tip_percentage
# from pyspark.sql.functions import col, when, round as spark_round

# trips_enriched = trips_enriched.withColumn(
#     "tip_percentage",
#     when(col("fare_amount") != 0, (col("tip_amount") / col("fare_amount")) * 100)
#     .otherwise(0)
# )

# trips_enriched.select("tip_amount", "fare_amount", "tip_percentage").show(5)

+----------+-----------+------------------+
|tip_amount|fare_amount|    tip_percentage|
+----------+-----------+------------------+
|       0.0|       17.7|               0.0|
|      3.75|       10.0|              37.5|
|       3.0|       23.3|12.875536480686694|
|       2.0|       10.0|              20.0|
|       3.2|        7.9| 40.50632911392405|
+----------+-----------+------------------+
only showing top 5 rows


In [38]:
trips_enriched = trips_valid.withColumns({
    'trip_duration_min': F.round(
        (F.unix_timestamp('dropoff_time') - F.unix_timestamp('pickup_time')) / 60, 2
    ),
    'trip_speed_mph': F.when(
        (F.unix_timestamp('dropoff_time') - F.unix_timestamp('pickup_time')) / 3600 > 0,
        F.round(F.col('trip_distance') / ((F.unix_timestamp('dropoff_time') - F.unix_timestamp('pickup_time')) / 3600), 2)
    ).otherwise(0),
    'pickup_hour': F.hour('pickup_time'),
    'pickup_day': F.dayofweek('pickup_time'),
    'tip_percentage': F.when(
        F.col('fare_amount') != 0,
        F.round((F.col('tip_amount') / F.col('fare_amount')) * 100, 2)
    ).otherwise(0)
})

print(f'Final enriched rows: {trips_enriched.count():,}')
print(f'\nTotal rows removed: {trips.count() - trips_enriched.count():,}')

trips_enriched.show(5)

Final enriched rows: 2,870,102

Total rows removed: 94,522
+-------------------+-------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+-----------------+--------------+-----------+----------+--------------+
|        pickup_time|       dropoff_time|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|PULocationID|DOLocationID|trip_duration_min|trip_speed_mph|pickup_hour|pickup_day|tip_percentage|
+-------------------+-------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+-----------------+--------------+-----------+----------+--------------+
|2024-01-01 00:57:55|2024-01-01 01:17:43|              1|         1.72|       17.7|       0.0|        22.7|           2|         186|          79|             19.8|          5.21|          0|         2|           0.0|
|2024-01-01 00:03:00|2024-01-01 00:09:36|              1|          1.

### Spark SQL Analytics

In [39]:
# Register your cleaned DataFrame as a temporary SQL view

# Register as a temporary view
trips_enriched.createOrReplaceTempView('trips')

# Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip percentage for each?
busiest_locations = spark.sql('''
    SELECT pickup_hour,
    COUNT(*) as num_trips,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(AVG(tip_percentage), 2) as avg_tip_percentage
    FROM trips
    GROUP BY pickup_hour
    ORDER BY num_trips DESC
    LIMIT 10
''')

print('Top 10 Busiest Pickup Hours:')
busiest_locations.show()

Top 10 Busiest Pickup Hours:
+-----------+---------+--------+------------------+
|pickup_hour|num_trips|avg_fare|avg_tip_percentage|
+-----------+---------+--------+------------------+
|         18|   206284|   17.01|             22.78|
|         17|   200315|   18.12|             22.34|
|         16|   184971|   19.46|             21.83|
|         15|   184009|   19.11|              19.8|
|         19|   178812|   17.63|             22.86|
|         14|   178031|   19.27|              19.8|
|         13|   165361|   18.42|             19.78|
|         12|   159916|    17.8|             19.74|
|         21|   155915|   18.29|             21.88|
|         20|   155561|   18.05|             22.17|
+-----------+---------+--------+------------------+



Interpretation: The top 10 busiest pickup hours were mostly hours in the evening when person may be commuting from work or engaging in nightlife activities. However, some hours where in the afternoon (12-14) indicating daytime demand as well which may likely be due to lunch-time travel/errands.

In [44]:
# Query 2: Which day of the week has the highest average trip speed? Include average distance and duration.
highest_speed = spark.sql(''' 
    SELECT pickup_day,
    ROUND(AVG(trip_speed_mph), 2) as avg_speed_mph,
    ROUND(AVG(trip_distance), 2) as avg_distance,
    ROUND(AVG(trip_duration_min), 2) as avg_duration_minutes
    FROM trips
    GROUP BY pickup_day
    ORDER BY avg_speed_mph DESC
    LIMIT 1
''')

print('Day of the Week with Highest Average Trip Speed:')
highest_speed.show()

Day of the Week with Highest Average Trip Speed:
+----------+-------------+------------+--------------------+
|pickup_day|avg_speed_mph|avg_distance|avg_duration_minutes|
+----------+-------------+------------+--------------------+
|         3|        17.46|        4.25|               16.17|
+----------+-------------+------------+--------------------+



Interpretation: The highest average speed trip was 17.46 mph on a Tuesday, which lasted for 16.17 minutes. This may be due to better travel condition i.e less traffic compared to the other days in NYC.

In [50]:
# Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week.
from pyspark.sql.window import Window

# Aggregate revenue by day and pickup location
daily_location_revenue = trips_enriched \
    .groupBy("pickup_day", "PULocationID") \
    .agg(F.sum("total_amount").alias("total_revenue"))

# Define window for ranking locations within each day
location_window = Window.partitionBy('pickup_day').orderBy(F.desc('total_revenue'))

# Rank locations and calculate percentage of that day's max revenue
ranked_trips = daily_location_revenue \
    .withColumn('revenue_rank', F.row_number().over(location_window)) \
    .withColumn('day_max_revenue', F.max('total_revenue').over(Window.partitionBy('pickup_day'))) \
    .withColumn('pct_of_day_max', F.round(F.col('total_revenue') / F.col('day_max_revenue') * 100, 2))

# Show top 5 pickup locations for each day
ranked_trips \
    .filter(F.col('revenue_rank') <= 5) \
    .select('pickup_day', 'PULocationID', 'total_revenue', 'revenue_rank', 'pct_of_day_max') \
    .orderBy('pickup_day', 'revenue_rank') \
    .show(35)

+----------+------------+------------------+------------+--------------+
|pickup_day|PULocationID|     total_revenue|revenue_rank|pct_of_day_max|
+----------+------------+------------------+------------+--------------+
|         1|         132|1564287.9299999834|           1|         100.0|
|         1|         138| 763398.5400000012|           2|          48.8|
|         1|         230| 346553.9500000004|           3|         22.15|
|         1|         186|         264131.38|           4|         16.89|
|         1|          79|263467.74000000034|           5|         16.84|
|         2|         132|2054606.7299999362|           1|         100.0|
|         2|         138|1021138.2800000039|           2|          49.7|
|         2|         161| 460145.2800000035|           3|          22.4|
|         2|         236| 373008.8900000014|           4|         18.15|
|         2|         237|  372575.480000002|           5|         18.13|
|         3|         132|1795093.5599999563|       

Interpretation: For all days of the week the PULocationID 132 was noted to be the top revenue generating pickup zone i.e it was always ranked no. 1 and is set as the daily baseline. PULocation 138 was consistently ranked no. 2 as well, where it usually generated about 39-61% of the top zone's revenue, while the other locations contributed smaller amounts. 

In [51]:
# Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips?

# Aggregate to hourly level first
hourly_trip = trips_enriched.groupBy('pickup_hour').agg(
    F.count('*').alias('hourly_trips')
).orderBy('pickup_hour')

# Window for cumulative sum (all rows up to current)
cumulative_window = Window.orderBy('pickup_hour').rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

#calculate cumulative count
hourly_trip = hourly_trip.withColumn(
    'cumulative_trips',
    F.sum('hourly_trips').over(cumulative_window)
)

# Calculate total trips to find 50% threshold
total_trips = hourly_trip.agg(F.sum('hourly_trips')).collect()[0][0]
threshold = total_trips * 0.5

# Find the hour where cumulative trips surpass 50% of total
first_hour = hourly_trip.filter(F.col('cumulative_trips') >= threshold) \
    .orderBy('pickup_hour') \
    .select('pickup_hour') \
    .first()[0]

print(f"Cumulative trips surpass 50% at hour {first_hour}")
hourly_trip.show(24)

Cumulative trips surpass 50% at hour 15
+-----------+------------+----------------+
|pickup_hour|hourly_trips|cumulative_trips|
+-----------+------------+----------------+
|          0|       75251|           75251|
|          1|       50491|          125742|
|          2|       34976|          160718|
|          3|       22948|          183666|
|          4|       15285|          198951|
|          5|       17496|          216447|
|          6|       39415|          255862|
|          7|       80872|          336734|
|          8|      113508|          450242|
|          9|      125621|          575863|
|         10|      135425|          711288|
|         11|      146754|          858042|
|         12|      159916|         1017958|
|         13|      165361|         1183319|
|         14|      178031|         1361350|
|         15|      184009|         1545359|
|         16|      184971|         1730330|
|         17|      200315|         1930645|
|         18|      206284|         2

Interpretation: The cumulative distribution shows that 50% of all trips is reached by hour 15 (3 PM), meaning half of daily taxi demand occurs from midnight through mid‑afternoon. Trip volume is lowest overnight (about 2–5 AM), rises steadily in the morning, and peaks in the late afternoon/evening (around 5–6 PM), indicating demand is concentrated in daytime and commuter hours.